In [1]:
import pandas as pd
import yaml
import re
import os
DB_FOLDER = f"dataset\database_v3"
os.makedirs(DB_FOLDER, exist_ok=True)
SPLIT = 241
config_path = os.path.join(DB_FOLDER, "config.yaml")
# 1. Load existing config (if exists)
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

config["split"] = SPLIT

In [2]:
macro_df = pd.read_csv(os.path.join(DB_FOLDER, "macro_data.csv"))
macro_feature_cols = macro_df.columns.to_list()[2:]
# 1. Train split (DO NOT fill yet)
train_mask = macro_df['timestep'] <= 265
train_df = macro_df.loc[train_mask, macro_feature_cols]

# 2. Compute scaler from TRAIN only (NaN ignored automatically)
min_vals = train_df.min()
max_vals = train_df.max()
range_vals = (max_vals - min_vals).replace(0, 1)

# 3. Normalize ALL data (NaN stays NaN)
macro_df_norm = macro_df.copy()
macro_df_norm[macro_feature_cols] = (
    2 * (macro_df[macro_feature_cols] - min_vals) / range_vals - 1
)

# 4. Fill NaN AFTER normalization
macro_df_norm[macro_feature_cols] = macro_df_norm[macro_feature_cols].fillna(0)
macro_df_norm.to_csv(os.path.join(DB_FOLDER, "macro_data_processed.csv"), index=False)

# 4. Store normalization info
macro_norm = {
    "method": "minmax_-1_1",
    "train_timestep_max": 265,
    "features": {}
}

for col in macro_feature_cols:
    macro_norm["features"][col] = {
        "min": float(min_vals[col]),
        "max": float(max_vals[col])
    }

# 5. Update config (add or overwrite macro section)
config["macro_normalization"] = macro_norm

In [3]:
project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))

# --------------------------------------------------
# 1. normalize selected numeric columns to [-1, 1]
#    using ALL projects, then fillna(0)
# --------------------------------------------------
norm_cols = ['latitude', 'longitude', 'Large_Dev_200plus', 'Condo_Age_2026']

min_vals = project_df[norm_cols].min()
max_vals = project_df[norm_cols].max()
range_vals = (max_vals - min_vals).replace(0, 1)

project_df[norm_cols] = 2 * (project_df[norm_cols] - min_vals) / range_vals - 1
project_df[norm_cols] = project_df[norm_cols].fillna(0)

# --------------------------------------------------
# 2. drop address columns
# --------------------------------------------------
project_df = project_df.drop(columns=['onemap_address', 'Street Name'], errors='ignore')

# --------------------------------------------------
# 3. one-hot encode categorical columns as 0/1
# --------------------------------------------------
cat_cols = ['Postal District', 'Property Type', 'Planning Region', 'Planning Area']

project_df = pd.get_dummies(
    project_df,
    columns=cat_cols,
    prefix=cat_cols,
    dummy_na=False,
    dtype='int8'
)

# --------------------------------------------------
# 4. process tenure_remaining_years
#    clip at 100, normalize to [-1, 1], then fillna(0)
# --------------------------------------------------
project_df['tenure_remaining_years'] = project_df['tenure_remaining_years'].clip(upper=100)

tenure_min = project_df['tenure_remaining_years'].min()
tenure_max = project_df['tenure_remaining_years'].max()
tenure_range = tenure_max - tenure_min
if pd.isna(tenure_range) or tenure_range == 0:
    tenure_range = 1

project_df['tenure_remaining_years'] = (
    2 * (project_df['tenure_remaining_years'] - tenure_min) / tenure_range - 1
)
project_df['tenure_remaining_years'] = project_df['tenure_remaining_years'].fillna(0)

# --------------------------------------------------
# save processed project file
# --------------------------------------------------
project_df = project_df.sort_values('project_id').reset_index(drop=True)
project_df.to_csv(os.path.join(DB_FOLDER, "project_processed.csv"), index=False)

# --------------------------------------------------
# save normalization info to existing config.yaml
# --------------------------------------------------
config_path = os.path.join(DB_FOLDER, "config.yaml")

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

project_norm = {
    "method": "minmax_-1_1",
    "features": {}
}

for col in norm_cols:
    project_norm["features"][col] = {
        "min": float(min_vals[col]),
        "max": float(max_vals[col])
    }

project_norm["features"]["tenure_remaining_years"] = {
    "clip_max": 100.0,
    "min": float(tenure_min),
    "max": float(tenure_max)
}

config["project_normalization"] = project_norm

In [4]:
school_df = pd.read_csv(os.path.join(DB_FOLDER, "school.csv"))
school_cols = school_df.columns.to_list()[2:]# fill value
fill_value = 2
school_df[school_cols] = school_df[school_cols].fillna(fill_value)
school_df.to_csv(os.path.join(DB_FOLDER, "school_processed.csv"), index=False)
config["school_processing"] = {
    "fillna_value": fill_value,
    "columns": school_cols
}

In [5]:
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "mrt.csv"))
mrt_cols = mrt_df.columns.to_list()[2:]
fill_value = 5 
mrt_df[mrt_cols] = mrt_df[mrt_cols].clip(upper=fill_value).fillna(fill_value)
mrt_df.to_csv(os.path.join(DB_FOLDER, "mrt_processed.csv"), index=False)
config["mrt_processing"] = {
    "fillna_value": fill_value,
    "columns": mrt_cols
}

In [6]:
dist_df = pd.read_csv(os.path.join(DB_FOLDER, "proximity.csv"))
dist_cols = dist_df.columns.to_list()[2:]
fill_value = 10
dist_df[dist_cols] = dist_df[dist_cols].clip(upper=fill_value).fillna(fill_value)
dist_df.to_csv(os.path.join(DB_FOLDER, "proximity_processed.csv"), index=False)
config["proximity_processing"] = {
    "fillna_value": fill_value,
    "columns": dist_cols
}

In [7]:
with open(config_path, "w") as f:
    yaml.dump(config, f, sort_keys=False)